# LAB 2 — Medical Image Segmentation
**Course:** Medical Image Analysis · Biomedical Engineering · URJC 2026/2027
**Group:** TODO (e.g. G2_x) · **Authors:** TODO · **Date:** TODO

**Executive summary (TODO — write it when the lab is finished):** images used, ROIs, methods compared, main conclusion.

---
### Notebook outline
1. Environment Setup & Project Verification
2. Clinical Data Loading & ROI Definition
3. Intensity-Based Segmentation — Thresholding
4. Region-Based Segmentation — Region Growing
5. Gradient & Morphological Segmentation — Watershed
6. Deformable Models — Active Contours (Snakes)
7. Pre- and Post-Processing (histogram, filtering, morphology)
8. Part 3 — Additional Segmentation Method
9. Ground Truth Integration & Quantitative Evaluation (DSC, Hausdorff, SSIM)
10. Exporting Final Segmentation to `.seg.nrrd`
11. Team Contributions & AI Disclosure


## Section 1 — Environment Setup & Project Verification

**To include**
- Imports (`numpy`, `scipy`, `scikit-image`, `matplotlib`, `pandas`, `pynrrd`) and library versions.
- Project paths: `DATA_RAW`, `DATA_GT`, `RESULTS_FIGURES`, `RESULTS_TABLES`, `RESULTS_SEG`.
- Random seed for reproducibility.
- Check that the two Lab 1 images exist in `raw_images/`.

**Functions:** `utils.ensure_dirs`, `utils.set_seed`


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import scipy
import skimage
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import nrrd

# Make the project root (the folder that contains src/) importable, whatever the working dir
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in (_cwd, *_cwd.parents, _cwd / "LAB2_MIA")
                    if (p / "src" / "utils.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import (active_contours, advanced_methods, export_seg, metrics,
                 preprocessing, region_growing, thresholding, utils, watershed_seg)

# Project paths (defined once in src/utils.py)
DATA_RAW = utils.DATA_RAW
DATA_GT = utils.DATA_GT
RESULTS_FIGURES = utils.RESULTS_FIGURES
RESULTS_TABLES = utils.RESULTS_TABLES
RESULTS_SEG = utils.RESULTS_SEG

# Output folders and reproducibility
utils.ensure_dirs()
SEED = utils.set_seed(42)

# Library versions
print(f"numpy {np.__version__} | scipy {scipy.__version__} | scikit-image {skimage.__version__}")
print(f"matplotlib {matplotlib.__version__} | pandas {pd.__version__} | pynrrd {nrrd.__version__}")
print(f"Random seed: {SEED}")

# Project verification
print(f"\nProject root: {PROJECT_ROOT}")
for folder in (DATA_RAW, DATA_GT, RESULTS_FIGURES, RESULTS_TABLES, RESULTS_SEG):
    print(f"  {'OK     ' if folder.exists() else 'MISSING'} {folder.relative_to(PROJECT_ROOT)}")

EXPECTED_IMAGES = ["Brain_MRI_131749_T1.png", "Chest_Xray_PA_3-8-2010.png"]
for name in EXPECTED_IMAGES:
    path = DATA_RAW / name
    print(f"  {'OK     ' if path.exists() else 'MISSING'} {path.relative_to(PROJECT_ROOT)}")
missing = [name for name in EXPECTED_IMAGES if not (DATA_RAW / name).exists()]
assert not missing, f"Missing images in {DATA_RAW}: {missing}"

## Section 2 — Clinical Data Loading & ROI Definition

We reuse the **two images from Lab 1**, which show different anatomical regions:

| Image | Modality | Anatomical structure segmented (ROI) |
|---|---|---|
| `Brain_MRI_131749_T1.png` | Axial T1-weighted brain MRI | Focal **hypointense lesion** in the frontal white matter |
| `Chest_Xray_PA_3-8-2010.png` | Posteroanterior (PA) chest X-ray | **Right lung** field (left side of the image, radiological convention) |

### Why define a Region of Interest (ROI) first?
Before segmenting, each image is cropped to a rectangular **bounding box** $I_{\text{ROI}} = I[y_{\min}:y_{\max},\, x_{\min}:x_{\max}]$ (rows = $y$, columns = $x$) that contains the target structure and a small margin of surrounding tissue:

- **Computational cost.** Every algorithm in this lab (region growing, watershed, snakes, Part 3) scales with the number of pixels processed. The ROI keeps only $N_{\text{ROI}} = (y_{\max}-y_{\min})\,(x_{\max}-x_{\min})$ pixels instead of the whole image (≈ 3 % of the MRI and ≈ 30 % of the X-ray, see table below).
- **Fewer distractors.** Outside the ROI there are structures with similar intensity or strong edges that are not of interest: skull, scalp and the other hemisphere in the MRI; the mediastinum, the other lung, the clavicles, the abdomen and the burned-in labels ("L", "ETB") in the X-ray. Removing them makes the histogram of the ROI much closer to bimodal (object vs. surrounding tissue), which helps thresholding, and prevents region growing/watershed from *leaking* into, or snakes from being attracted by, unrelated edges.
- **Reproducibility.** The ROI and the seed points are fixed numerically below and reused, unchanged, in all later sections, so that every method is compared on exactly the same pixels. The final masks are re-embedded in the original image size before exporting them to `.seg.nrrd` (Section 10).

The seed points are written in **full-image coordinates** `(row, col)` (easy to read on the overview figure) and converted to ROI coordinates with `utils.to_roi_coords`. Foreground (FG) seeds lie inside the structure (used by region growing, watershed, snakes and Part 3); background (BG) seeds lie on the surrounding tissue (watershed markers and seeded Part 3 methods).

In [ ]:
# --- 2.1 Load the two Lab 1 images as 2-D float arrays in [0, 1] -------------------------
IMAGE_FILES = {
    "brain": "Brain_MRI_131749_T1.png",
    "chest": "Chest_Xray_PA_3-8-2010.png",
}
images = {key: utils.load_image(fname) for key, fname in IMAGE_FILES.items()}

for key, img in images.items():
    print(f"{key:6s} {IMAGE_FILES[key]:28s} shape={img.shape}  dtype={img.dtype}  "
          f"range=[{img.min():.3f}, {img.max():.3f}]  N={img.size} px")

In [ ]:
# --- 2.2 ROI (bounding box) and seed points per image -----------------------------------
# ROI = (ymin, ymax, xmin, xmax) -> img[ymin:ymax, xmin:xmax]; points are (row, col) in FULL-image coordinates
CASES = {
    "brain": {
        "structure": "Focal hypointense lesion (frontal white matter)",
        "roi": (120, 205, 125, 215),
        "fg_points": [(162, 168)],                                        # dark centre of the lesion
        "bg_points": [(135, 140), (135, 200), (190, 140), (190, 200)],   # surrounding white matter
    },
    "chest": {
        "structure": "Right lung field",
        "roi": (180, 1800, 230, 1100),
        "fg_points": [(700, 600), (1000, 550), (1300, 700)],              # upper / middle / lower lung
        "bg_points": [(1000, 1080), (1500, 1080), (1000, 260), (1780, 600)],  # mediastinum, heart, chest wall, diaphragm
    },
}

rois, seeds, bg_seeds = {}, {}, {}
summary = []
for key, case in CASES.items():
    img, box = images[key], case["roi"]
    rois[key] = utils.crop_roi(img, box)
    seeds[key] = utils.to_roi_coords(case["fg_points"], box)      # ROI coordinates, reused later
    bg_seeds[key] = utils.to_roi_coords(case["bg_points"], box)   # ROI coordinates, reused later
    h, w, n_roi = utils.roi_dimensions(box)
    summary.append({
        "Image": IMAGE_FILES[key], "Structure (ROI)": case["structure"],
        "Bounding box [ymin:ymax, xmin:xmax]": f"[{box[0]}:{box[1]}, {box[2]}:{box[3]}]",
        "Height (px)": h, "Width (px)": w, "N_ROI (px)": n_roi,
        "% of image": round(100 * n_roi / img.size, 2),
        "FG seeds (ROI coords)": seeds[key], "BG seeds (ROI coords)": bg_seeds[key],
    })

roi_table = pd.DataFrame(summary).set_index("Image")
utils.save_table(roi_table, "02_roi_definition.csv")
roi_table

In [ ]:
# --- 2.3 Sanity check: intensity at the seeds (FG should differ clearly from BG) ---------
for key in CASES:
    roi_img = rois[key]
    fg_vals = [roi_img[r, c] for r, c in seeds[key]]
    bg_vals = [roi_img[r, c] for r, c in bg_seeds[key]]
    print(f"{key:6s} ROI shape={roi_img.shape} | FG mean = {np.mean(fg_vals):.3f} "
          f"(values {np.round(fg_vals, 3).tolist()}) | BG mean = {np.mean(bg_vals):.3f} "
          f"(values {np.round(bg_vals, 3).tolist()})")

In [ ]:
# --- 2.4 Comparison figure: full image with the red ROI box vs. cropped ROI (+ seeds) ---
for i, key in enumerate(CASES):
    fig = utils.plot_roi_overview(images[key], CASES[key]["roi"], title=CASES[key]["structure"])
    ax_full, ax_roi = fig.axes
    ymin, ymax, xmin, xmax = CASES[key]["roi"]
    fg = np.array(CASES[key]["fg_points"]); bg = np.array(CASES[key]["bg_points"])
    ax_full.plot(fg[:, 1], fg[:, 0], "g+", markersize=10, markeredgewidth=2)
    fg_roi = np.array(seeds[key]); bg_roi = np.array(bg_seeds[key])
    ax_roi.plot(fg_roi[:, 1], fg_roi[:, 0], "g+", markersize=12, markeredgewidth=2, label="FG seeds")
    ax_roi.plot(bg_roi[:, 1], bg_roi[:, 0], "bx", markersize=8, markeredgewidth=2, label="BG seeds")
    ax_roi.legend(loc="lower right", fontsize=8)
    path = utils.save_figure(fig, f"02_roi_overview_{key}.png")
    print(f"Saved: {path.relative_to(PROJECT_ROOT)}")
    plt.show()

## Section 3 — Intensity-Based Segmentation: Thresholding

**Markdown to write:** global thresholding (Otsu) vs. dual/manual thresholding chosen from the histogram (bimodal tissue separation).

**To include**
- ROI histogram with the threshold line(s) $T$.
- Binary mask from Otsu and from a manual/dual threshold.
- **Parameter study:** effect of $T$ on the mask (assignment requires analysing adjustable parameters).

**Functions:** `thresholding.otsu`, `thresholding.multi_otsu`, `thresholding.apply_threshold`, `thresholding.plot_histogram`, `utils.show_overlays`

**Discussion (students):** TODO


## Section 4 — Region-Based Segmentation: Region Growing

**Markdown to write:** region growing from seeds; impact of neighbourhood connectivity (4 vs. 8 in 2-D; 6 vs. 26 in 3-D) on the risk of *leakage* into adjacent structures.

**To include**
- Manual seed $(x_0, y_0)$ and intensity tolerance.
- 4-connected vs. 8-connected comparison.
- Printed region size (number of pixels).
- **Parameter study:** tolerance (and seed position).

**Functions:** `region_growing.region_growing`, `region_growing.compare_connectivity`, `utils.show_overlays`

**Discussion (students):** TODO


## Section 5 — Gradient & Morphological Segmentation: Watershed

**Markdown to write:** watershed as flooding of a topographic relief (intensity/gradient = height); why foreground/background markers mitigate over-segmentation.

**To include**
- Gradient map (Sobel and/or morphological gradient).
- Foreground and background markers.
- Flooding result and the resulting contour.
- Result **without markers** (over-segmentation) for comparison.
- **Parameter study:** gradient type, marker size/position.

**Functions:** `watershed_seg.gradient_map`, `watershed_seg.build_markers`, `watershed_seg.run_watershed`, `watershed_seg.watershed_oversegmentation`, `utils.show_overlays`

**Discussion (students):** TODO


## Section 6 — Deformable Models: Active Contours (Snakes)

**Markdown to write:** short summary of the snake model; internal forces ($\alpha$ elasticity, $\beta$ rigidity) and external forces (attraction to edges/lines).
*Note for the text:* in scikit-image the external forces are weighted by `w_edge` and `w_line`; `gamma` is the explicit time step of the evolution, not an edge-attraction weight.

**To include**
- Initial curve (circle/ellipse) near the anatomical boundary.
- Iterative evolution until convergence (initial vs. final contour).
- Binary mask from the final contour.
- **Parameter study:** $\alpha$, $\beta$, `w_edge`, `sigma`, initial curve.

**Functions:** `active_contours.circle_init`, `active_contours.run_snake`, `active_contours.snake_to_mask`

**Discussion (students):** TODO


## Section 7 — Pre- and Post-Processing

*Required by the assignment:* discuss whether the previous segmentations can be improved with pre-/post-processing and **support the hypothesis with experiments** on your images.

**To include**
- Histogram modifications (equalization, CLAHE, contrast stretching).
- Filtering (Gaussian, median, bilateral).
- Morphological post-processing of the masks (opening, closing, hole filling, small-object removal).
- Before/after comparison (figures and/or table).

**Functions:** `preprocessing.equalize_hist`, `preprocessing.clahe`, `preprocessing.contrast_stretch`, `preprocessing.smooth_gaussian`, `preprocessing.smooth_median`, `preprocessing.smooth_bilateral`, `preprocessing.clean_mask`

**Discussion (students):** TODO


## Section 8 — Part 3: Additional Segmentation Method

*Required by the assignment:* in-depth explanation of **one** state-of-the-art method of your choice, its implementation/code source, evaluation on the same images, and comparison with the previous methods.

**To include**
- Chosen method: TODO (suggestions: Chan–Vese, Random Walker, graph cuts, k-means/GMM, a pretrained deep model).
- Explanation of the method and its parameters (markdown, in your own words).
- Code source / reference.
- Results on both images and a parameter study.

**Functions:** `advanced_methods.chan_vese_segmentation` *or* `advanced_methods.random_walker_segmentation` (keep only the one you use)

**Discussion (students):** TODO


## Section 9 — Ground Truth Integration & Quantitative Evaluation

**Markdown to write:** why each metric is used:
- **Dice Similarity Coefficient (DSC):** spatial overlap between areas/volumes.
- **Hausdorff Distance (HD):** largest distance between the boundaries of the segmentation and the reference.
- **SSIM:** structural similarity between the binary masks.

**To include**
- Load the manual ground truth made in 3D Slicer (`.seg.nrrd`) or Paint (`.png`) from `data/ground_truth/`.
- Comparison table with the 3 metrics for **every** method (Thresholding, Region Growing, Watershed, Snakes, Part 3 method), for both images.
- Export to `results/tables/02_segmentation_metrics_summary.csv`.
- Overview figure of all methods vs. ground truth.

**Functions:** `export_seg.read_mask`, `metrics.dice`, `metrics.hausdorff`, `metrics.ssim_binary`, `metrics.evaluate_methods`, `utils.save_table`, `utils.show_overlays`

**Method choice per ROI (students):** TODO — justify which algorithm you choose for each anatomical structure.


## Section 10 — Exporting Final Segmentation to `.seg.nrrd`

**Markdown to write:** what the `.seg.nrrd` format is and why 3D Slicer and clinical planning software use it (segment name, colour, label value, extent in the header).

**To include**
- Re-embed the ROI mask into the dimensions of the original image.
- Write the file with `nrrd.write()` and a proper header (segment name, colour, dimensions).
- Read it back and check shape/overlay.

**Functions:** `export_seg.embed_roi_mask`, `export_seg.write_seg_nrrd`, `export_seg.read_mask`


## Section 11 — Team Contributions & AI Disclosure

*The report must clearly state both (assignment requirement).*

**Contribution of each team member:** TODO

**AI Disclosure Statement:** TODO — which tool, in which sections and for what purpose. AI may assist with code implementation but must not replace your programming work or judgment, nor be used for the interpretation/discussion of results.

**Final checklist:** 2 images · ROI justified · 3 base algorithms + snakes + Part 3 · parameter influence · method choice per ROI · pre/post-processing hypothesis · report ≤ 10 pages · zip named `P2_MIA_BED_26_27_GroupX.zip`.
